# Qdrant To Kayak Reranking

This notebook shows a complete local path:

1. store ColBERT-style token matrices in Qdrant
2. filter candidates in Qdrant
3. fetch candidate vectors with `with_vectors=True`
4. rerank those candidates exactly in Kayak with the Mojo backend
5. inspect clause-text verification
6. measure local average timings

Install for this notebook with:

```bash
uv add qdrant-client
```

or:

```bash
pixi add --pypi qdrant-client
```

In [1]:
import time
import warnings

import numpy as np
import kayak
from colbert.infra.config import ColBERTConfig
from colbert.modeling.checkpoint import Checkpoint
import torch

DEFAULT_MODEL_NAME = "colbert-ir/colbertv2.0"

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="CUDA is not available.*", category=UserWarning)
warnings.filterwarnings(
    "ignore",
    message="torch.cuda.amp.GradScaler is enabled, but CUDA is not available.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="resource_tracker: There appear to be .* leaked semaphore objects.*",
    category=UserWarning,
)

torch.set_num_threads(1)
torch.multiprocessing.set_sharing_strategy("file_system")

checkpoint = Checkpoint(
    DEFAULT_MODEL_NAME,
    colbert_config=ColBERTConfig(gpus=0),
    verbose=0,
)

def encode_query_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.queryFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()


def encode_document_text(text: str) -> list[list[float]]:
    with torch.inference_mode():
        encoded = checkpoint.docFromText([text], to_cpu=True)
    return encoded[0].detach().cpu().tolist()

warnings.filterwarnings("ignore", category=DeprecationWarning)

BACKEND = kayak.MOJO_EXACT_CPU_BACKEND

def timed_avg_ms(fn, repeats=50, warmup=5):
    for _ in range(warmup):
        fn()
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) * 1000.0 / repeats

def describe_hits(hits):
    rows = []
    for hit in hits:
        row = {
            "doc_id": hit.doc_id,
            "score": round(float(hit.score), 3),
        }
        clause_text = getattr(hit, "clause_text", None)
        if clause_text:
            row["text"] = clause_text
        rows.append(row)
    return rows

DOCS = [
    {
        "doc_id": "pixi_mojo_project",
        "topic": "installation",
        "text": "Create one Pixi project, install Python, Mojo, and kayak in the same environment, then set BACKEND = kayak.MOJO_EXACT_CPU_BACKEND in your application code.",
    },
    {
        "doc_id": "pixi_python_pin",
        "topic": "installation",
        "text": "Pixi can pin Python 3.11 and add PyPI packages together with Mojo in a single project environment.",
    },
    {
        "doc_id": "uv_kayak_only",
        "topic": "installation",
        "text": "uv add kayak installs the Python package but does not install the Mojo CLI, so the NumPy reference backend remains the only available backend.",
    },
    {
        "doc_id": "qdrant_multivector",
        "topic": "vector_db",
        "text": "Qdrant can store ColBERT-style multivectors and search them with a MAX_SIM comparator in one collection.",
    },
    {
        "doc_id": "weaviate_multivector",
        "topic": "vector_db",
        "text": "Weaviate embedded mode can store self-provided multivectors under a named vector such as colbert.",
    },
    {
        "doc_id": "lancedb_multivector",
        "topic": "vector_db",
        "text": "LanceDB can store multivector columns as list-of-list float data for local reranking experiments.",
    },
    {
        "doc_id": "clause_text_verifier",
        "topic": "search_plan",
        "text": "Kayak clause_text stage 3 verification returns supporting text for the final hits.",
    },
    {
        "doc_id": "stage_profiles",
        "topic": "search_plan",
        "text": "Kayak search plans expose candidate counts, stage 2 work, and verifier materialization counts explicitly.",
    },
]

query_text = "How should a project install Mojo and make Kayak use the Mojo backend by default?"
query_vectors = np.asarray(encode_query_text(query_text), dtype=np.float32)
query = kayak.query(query_vectors, text=query_text)

doc_ids = [doc["doc_id"] for doc in DOCS]
doc_texts = [doc["text"] for doc in DOCS]
doc_vectors = [np.asarray(encode_document_text(doc["text"]), dtype=np.float32) for doc in DOCS]

full_index = kayak.documents(doc_ids, doc_vectors, texts=doc_texts).pack()

{
    "query_vector_count": int(query_vectors.shape[0]),
    "document_vector_counts": {
        doc_id: int(vectors.shape[0])
        for doc_id, vectors in zip(doc_ids, doc_vectors)
    },
}

/tmp/kayak-vdb-tools/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Apr 15, 12:46:13] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


{'query_vector_count': 32,
 'document_vector_counts': {'pixi_mojo_project': 43,
  'pixi_python_pin': 26,
  'uv_kayak_only': 36,
  'qdrant_multivector': 29,
  'weaviate_multivector': 24,
  'lancedb_multivector': 26,
  'clause_text_verifier': 19,
  'stage_profiles': 25}}

## Create A Local Qdrant Collection

In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient(':memory:')
client.create_collection(
    collection_name='docs',
    vectors_config=models.VectorParams(
        size=128,
        distance=models.Distance.COSINE,
        multivector_config=models.MultiVectorConfig(
            comparator=models.MultiVectorComparator.MAX_SIM,
        ),
    ),
)

points = []
for idx, (doc, vectors) in enumerate(zip(DOCS, doc_vectors)):
    points.append(
        models.PointStruct(
            id=idx,
            vector=vectors.tolist(),
            payload={
                'doc_id': doc['doc_id'],
                'topic': doc['topic'],
                'text': doc['text'],
            },
        )
    )

client.upsert(collection_name='docs', points=points)
client.count(collection_name='docs').count

8

## Stage 1 Retrieval In Qdrant

In [3]:
def qdrant_stage1(limit=3):
    return client.query_points(
        collection_name='docs',
        query=query_vectors.tolist(),
        query_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key='topic',
                    match=models.MatchValue(value='installation'),
                )
            ]
        ),
        with_vectors=True,
        limit=limit,
    ).points

candidate_points = qdrant_stage1()
[(point.payload['doc_id'], round(float(point.score), 3)) for point in candidate_points]

[('pixi_mojo_project', 22.654),
 ('uv_kayak_only', 21.87),
 ('pixi_python_pin', 16.529)]

## Exact Kayak Rerank On Qdrant Candidates

In [4]:
def qdrant_candidate_index(points):
    return kayak.documents(
        [point.payload['doc_id'] for point in points],
        [np.asarray(point.vector, dtype=np.float32) for point in points],
        texts=[point.payload['text'] for point in points],
    ).pack()

candidate_index = qdrant_candidate_index(candidate_points)
reranked_hits = kayak.search(query, candidate_index, k=2, backend=BACKEND)
describe_hits(reranked_hits)

[{'doc_id': 'pixi_mojo_project', 'score': 22.654},
 {'doc_id': 'uv_kayak_only', 'score': 21.87}]

## Clause-Text Verification On The Same Candidates

In [5]:
verifier_plan = kayak.exact_full_scan_search_plan(
    final_k=2,
    candidate_k=len(candidate_points),
    stage3_verifier=kayak.clause_text_stage3_verifier_operator(),
)

verifier_result = kayak.search_with_plan(
    query,
    candidate_index,
    verifier_plan,
    backend=BACKEND,
)

print('stage3_hits:')
print(describe_hits(verifier_result.hits))
print('stage2_profile:', verifier_result.stage2)
print('stage3_profile:', verifier_result.stage3_verifier)

stage3_hits:
[{'doc_id': 'pixi_mojo_project', 'score': 25.654}, {'doc_id': 'uv_kayak_only', 'score': 24.27}]
stage2_profile: SearchStageProfile(stage_name='noop_topk', input_hit_count=3, output_hit_count=3, query_vector_count=0, document_count=3, document_vector_count=0, document_text_count=0, materialized_artifacts=())
stage3_profile: SearchStageProfile(stage_name='clause_text', input_hit_count=3, output_hit_count=2, query_vector_count=0, document_count=3, document_vector_count=0, document_text_count=3, materialized_artifacts=(StageArtifactMaterialization(family='document_text', document_count=3, document_vector_count=0, document_text_count=3),))


## Compare Against A Full Exact Kayak Baseline

In [6]:
exact_hits = kayak.search(query, full_index, k=2, backend=BACKEND)

comparison = {
    'qdrant_candidate_ids': [point.payload['doc_id'] for point in candidate_points],
    'kayak_reranked_ids': [hit.doc_id for hit in reranked_hits],
    'full_exact_ids': [hit.doc_id for hit in exact_hits],
}
comparison

{'qdrant_candidate_ids': ['pixi_mojo_project',
  'uv_kayak_only',
  'pixi_python_pin'],
 'kayak_reranked_ids': ['pixi_mojo_project', 'uv_kayak_only'],
 'full_exact_ids': ['pixi_mojo_project', 'uv_kayak_only']}

## Local Average Timings

In [7]:
def qdrant_plus_kayak():
    points = qdrant_stage1()
    idx = qdrant_candidate_index(points)
    return kayak.search(query, idx, k=2, backend=BACKEND)

timings_ms = {
    'qdrant_stage1_ms': timed_avg_ms(qdrant_stage1, repeats=100),
    'qdrant_plus_kayak_ms': timed_avg_ms(qdrant_plus_kayak, repeats=50),
    'full_kayak_exact_ms': timed_avg_ms(
        lambda: kayak.search(query, full_index, k=2, backend=BACKEND),
        repeats=100,
    ),
}

{name: round(value, 3) for name, value in timings_ms.items()}

{'qdrant_stage1_ms': 1.425,
 'qdrant_plus_kayak_ms': 2.921,
 'full_kayak_exact_ms': 0.773}